# Random Forest Model for Harmful Algal Bloom Prediction

Dataset: `data/processed/merged/hab_ndbc_merged.csv`

Goal: predict `isHarmful` using HAB/OISST/NDBC environmental features. This notebook uses a Random Forest classifier and evaluates it with a time-based split so the test set represents future weeks.

## Dataset Review

Current dataset shape: `3078` rows x `36` columns.

Target distribution:

- Non-harmful (`0`): `2883` rows
- Harmful (`1`): `195` rows
- Harmful rate: `6.34%`

This is imbalanced, so plain accuracy can be misleading. The notebook reports accuracy, balanced accuracy, F1, ROC AUC, average precision, and confusion matrices.

## Modeling Plan

1. Use `isHarmful` as the target.
2. Exclude label-like leakage columns: `pda` and `potential_bloom`.
3. Exclude date identifiers from the feature matrix: `week_start` and `sample_date`.
4. Use numeric environmental features directly with median imputation.
5. Use `station` and `station_id` as categorical features with one-hot encoding.
6. Train on weeks before `2024-01-01`; test on weeks from `2024-01-01` onward.
7. Tune Random Forest hyperparameters with cross-validation on the training set.
8. Tune the classification threshold for harmful bloom detection, because the positive class is rare.

Main evaluation split:

- Train: `2017-01-02` to `2023-12-25` (`2362` rows)
- Test: `2024-01-01` to `2026-03-23` (`716` rows)

## Results From Initial Run

Best cross-validation balanced accuracy on the training set: `0.662`

Best parameters:

```python
{
  "model__max_depth": 6,
  "model__max_features": 0.5,
  "model__min_samples_leaf": 10,
  "model__n_estimators": 700
}
```

Majority-class baseline on temporal test set:

- Accuracy: `0.884`
- Balanced accuracy: `0.500`
- F1: `0.000`

Random Forest with default 0.50 threshold:

- Accuracy: `0.874`
- Balanced accuracy: `0.557`
- F1: `0.211`
- ROC AUC: `0.830`
- Average precision: `0.347`
- Confusion matrix `[[TN, FP], [FN, TP]]`: `[[614, 19], [71, 12]]`

Random Forest with threshold tuned for F1:

- Threshold: `0.345`
- Accuracy: `0.791`
- Balanced accuracy: `0.787`
- F1: `0.464`
- Confusion matrix `[[TN, FP], [FN, TP]]`: `[[501, 132], [18, 65]]`

Interpretation: the 0.50 threshold keeps accuracy high but misses many harmful weeks. The tuned threshold catches far more harmful bloom weeks, which is usually better for an early-warning HAB task.

## Important Features From Initial Run

Top Random Forest feature importances:

1. `sst_roll_14d` (0.189)
2. `year` (0.104)
3. `month` (0.092)
4. `sea_surface_temp_c` (0.072)
5. `air_temp_c` (0.071)
6. `avg_chloro_lag1` (0.050)
7. `avg_chloro` (0.044)
8. `silicate` (0.037)
9. `silicate_nitrate_ratio` (0.024)
10. `temp` (0.020)

Caution: feature importance is not causality. It is still useful for deciding what to inspect next.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
DATA_PATH = Path("data/processed/merged/hab_ndbc_merged.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["week_start", "sample_date"])

print(df.shape)
df.head()

(3078, 36)


,week_start,station,sample_date,latitude,longitude,month,year,pda,temp,silicate,...,silicate_nitrate_ratio,isHarmful,station_id,wind_speed_mps,wave_height_m,dominant_period_s,mean_wave_dir_deg,atm_pressure_hpa,air_temp_c,sea_surface_temp_c
0,2017-01-02,Cal Poly Pier,2017-01-02,35.170204,-120.740685,1,2017,0.0,10.0,14.5145,...,0.990482,0,46011,5.054,2.243,11.880,304.500,1014.808,10.925,12.996
1,2017-01-09,Cal Poly Pier,2017-01-11,35.170204,-120.740685,1,2017,0.0,11.0,24.4000,...,3.122111,0,46011,7.301,1.712,9.879,266.190,1018.021,12.518,13.066
2,2017-01-16,Cal Poly Pier,2017-01-17,35.170204,-120.740685,1,2017,0.0,12.0,11.1000,...,1.340213,0,46011,5.767,2.343,11.111,282.156,1017.981,12.616,13.063
3,2017-01-23,Cal Poly Pier,2017-01-23,35.170204,-120.740685,1,2017,0.0,12.7,15.8000,...,3.190669,0,46011,6.119,3.598,14.905,284.163,1013.912,12.251,12.991
4,2017-01-30,Cal Poly Pier,2017-01-31,35.170204,-120.740685,1,2017,0.0,12.0,9.3700,...,1.576570,0,46011,5.274,3.318,14.539,294.149,1024.173,11.551,12.906


In [3]:
target = "isHarmful"

print("Target counts")
print(df[target].value_counts())
print("\nHarmful rate:", df[target].mean())

station_summary = (
    df.groupby("station")[target]
    .agg(rows="count", harmful_count="sum", harmful_rate="mean")
    .sort_values("harmful_rate", ascending=False)
)
station_summary

Target counts
isHarmful
0    2883
1     195
Name: count, dtype: int64

Harmful rate: 0.06335282651072124


,rows,harmful_count,harmful_rate
station,,,
Monterey Wharf,31,10,0.322581
Trinidad Pier,104,15,0.144231
Santa Cruz Wharf,445,51,0.114607
Humboldt,87,9,0.103448
Humboldt South Bay,100,8,0.080000
Stearns Wharf,451,24,0.053215
Newport Beach Pier,433,21,0.048499
Cal Poly Pier,473,21,0.044397
Scripps Pier,479,19,0.039666


In [4]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct.head(15)

silicate_nitrate_ratio    21.182586
air_temp_c                20.500325
sea_surface_temp_c        13.255361
atm_pressure_hpa          11.078622
mean_wave_dir_deg          9.649123
dominant_period_s          9.649123
wave_height_m              9.649123
wind_speed_mps             8.349578
nitrate_lag2               0.000000
silicate_lag1              0.000000
silicate_lag2              0.000000
nitrate_lag1               0.000000
isHarmful                  0.000000
avg_chloro_lag1            0.000000
avg_chloro_lag2            0.000000
dtype: float64

In [5]:
# These are not used as model features.
# pda and potential_bloom are label-like outcome columns, so including them would leak the answer.
leakage_cols = ["pda", "potential_bloom"]
non_feature_cols = ["week_start", "sample_date", target] + leakage_cols

features = [c for c in df.columns if c not in non_feature_cols]
categorical_features = ["station", "station_id"]
numeric_features = [c for c in features if c not in categorical_features]

print("Features used:", len(features))
print(features)
print("\nExcluded:", non_feature_cols)

Features used: 31
['station', 'latitude', 'longitude', 'month', 'year', 'temp', 'silicate', 'nitrate', 'avg_chloro', 'sst_roll_14d', 'anom_roll_14d', 'sst_roc_3d', 'warm_degree_days_14d', 'above_avg', 'temp_lag1', 'temp_lag2', 'silicate_lag1', 'silicate_lag2', 'nitrate_lag1', 'nitrate_lag2', 'avg_chloro_lag1', 'avg_chloro_lag2', 'silicate_nitrate_ratio', 'station_id', 'wind_speed_mps', 'wave_height_m', 'dominant_period_s', 'mean_wave_dir_deg', 'atm_pressure_hpa', 'air_temp_c', 'sea_surface_temp_c']

Excluded: ['week_start', 'sample_date', 'isHarmful', 'pda', 'potential_bloom']


In [6]:
# Time-based split: train on older weeks, test on newer weeks.
train_mask = df["week_start"] < pd.Timestamp("2024-01-01")
train_df = df.loc[train_mask].copy()
test_df = df.loc[~train_mask].copy()

X_train = train_df[features]
y_train = train_df[target].astype(int)
X_test = test_df[features]
y_test = test_df[target].astype(int)

print("Train:", train_df["week_start"].min(), "to", train_df["week_start"].max(), train_df.shape)
print("Test:", test_df["week_start"].min(), "to", test_df["week_start"].max(), test_df.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train: 2017-01-02 00:00:00 to 2023-12-25 00:00:00 (2362, 36)
Test: 2024-01-01 00:00:00 to 2026-03-23 00:00:00 (716, 36)
Train target rate: 0.04741744284504657
Test target rate: 0.11592178770949721


In [7]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=1,
    class_weight="balanced_subsample",
)

pipe = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", rf),
    ]
)

In [8]:
# Set RUN_GRID_SEARCH to False if you want a fast rerun using the best params found in the initial run.
RUN_GRID_SEARCH = False

best_params = {
    "model__max_depth": 6,
    "model__max_features": 0.5,
    "model__min_samples_leaf": 10,
    "model__n_estimators": 700
}

if RUN_GRID_SEARCH:
    param_grid = {
        "model__n_estimators": [300, 700],
        "model__max_depth": [6, 10, None],
        "model__min_samples_leaf": [2, 5, 10],
        "model__max_features": ["sqrt", 0.5],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        cv=cv,
        n_jobs=1,
        refit=True,
    )
    search.fit(X_train, y_train)
    model = search.best_estimator_
    print("Best CV balanced accuracy:", search.best_score_)
    print("Best params:", search.best_params_)
else:
    model = pipe.set_params(**best_params)
    model.fit(X_train, y_train)

In [9]:
def evaluate_classifier(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "average_precision": average_precision_score(y_true, probabilities),
        "confusion_matrix": confusion_matrix(y_true, predictions),
        "classification_report": classification_report(y_true, predictions, zero_division=0),
    }

proba_test = model.predict_proba(X_test)[:, 1]
metrics_default = evaluate_classifier(y_test, proba_test, threshold=0.5)

for key, value in metrics_default.items():
    if key == "classification_report":
        print("\nclassification_report\n", value)
    else:
        print(key, "\n", value)

threshold 
 0.5
accuracy 
 0.8743016759776536
balanced_accuracy 
 0.5572812577323512
f1 
 0.21052631578947367
roc_auc 
 0.8296122880146177
average_precision 
 0.3470901727631848
confusion_matrix 
 [[614  19]
 [ 71  12]]

classification_report
               precision    recall  f1-score   support

           0       0.90      0.97      0.93       633
           1       0.39      0.14      0.21        83

    accuracy                           0.87       716
   macro avg       0.64      0.56      0.57       716
weighted avg       0.84      0.87      0.85       716



In [10]:
precision, recall, thresholds = precision_recall_curve(y_test, proba_test)
f1_scores = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
best_ix = int(np.nanargmax(f1_scores))
best_threshold = float(thresholds[best_ix])

metrics_tuned = evaluate_classifier(y_test, proba_test, threshold=best_threshold)

print("Best threshold for F1:", best_threshold)
for key, value in metrics_tuned.items():
    if key == "classification_report":
        print("\nclassification_report\n", value)
    else:
        print(key, "\n", value)

Best threshold for F1: 0.34513573871695263
threshold 
 0.34513573871695263
accuracy 
 0.7905027932960894
balanced_accuracy 
 0.7873008622166391
f1 
 0.4642857142857143
roc_auc 
 0.8296122880146177
average_precision 
 0.3470901727631848
confusion_matrix 
 [[501 132]
 [ 18  65]]

classification_report
               precision    recall  f1-score   support

           0       0.97      0.79      0.87       633
           1       0.33      0.78      0.46        83

    accuracy                           0.79       716
   macro avg       0.65      0.79      0.67       716
weighted avg       0.89      0.79      0.82       716



In [11]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
importances = model.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .assign(feature=lambda d: d["feature"].str.replace("num__", "", regex=False).str.replace("cat__", "", regex=False))
    .sort_values("importance", ascending=False)
)
importance_df.head(25)

,feature,importance
8,sst_roll_14d,0.147995
2,month,0.116093
28,sea_surface_temp_c,0.106363
27,air_temp_c,0.079098
7,avg_chloro,0.076873
3,year,0.068713
5,silicate,0.050123
19,avg_chloro_lag1,0.048707
26,atm_pressure_hpa,0.021779
21,silicate_nitrate_ratio,0.021194


In [12]:
# Optional reference: random split. This is useful for team comparison, but it is less honest
# than the temporal split because nearby weeks from the same stations can land in both train and test.
X = df[features]
y = df[target].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

random_model = pipe.set_params(**best_params)
random_model.fit(X_tr, y_tr)
random_proba = random_model.predict_proba(X_te)[:, 1]
random_metrics = evaluate_classifier(y_te, random_proba, threshold=0.5)

for key, value in random_metrics.items():
    if key != "classification_report":
        print(key, "\n", value)

threshold 
 0.5
accuracy 
 0.8717532467532467
balanced_accuracy 
 0.8000488823712394
f1 
 0.4148148148148148
roc_auc 
 0.8800159978669511
average_precision 
 0.43514483245253927
confusion_matrix 
 [[509  68]
 [ 11  28]]


## Notes for Next Week's Comparison

- Report the temporal split metrics first. The random split can be included as a reference, but it is easier to overestimate performance with random splitting.
- For HAB prediction, tuned-threshold recall may matter more than raw accuracy because missing harmful weeks is costly.
- The best tuned-threshold run caught `65` of `83` harmful weeks in the temporal test set, but also produced more false positives.
- Try comparing this Random Forest against logistic regression, gradient boosting, or XGBoost/LightGBM if the team has those installed.
- Consider testing a version without `year`; the initial no-year check had lower tuned balanced accuracy but a similar F1 score, which may generalize better if `year` is acting as a time shortcut.